# GlucoTwin — Exploratory Data Analysis & Digital Twin Demonstration

This notebook demonstrates the synthetic data generation, feature engineering, model prediction, and Digital Twin state updates for **GlucoTwin**.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root in python path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.data.generate_synthetic import generate_and_save_data
from src.features.build_features import build_features_dataset
from src.twin.digital_twin import DigitalTwin

In [ ]:
# 1. Load or Generate Datasets
ehr_df, wearable_df = generate_and_save_data()
print(f"EHR Shape: {ehr_df.shape}")
print(f"Wearable Shape: {wearable_df.shape}")

In [ ]:
# 2. Inspect Patient State and Time Series
p_id = ehr_df.iloc[0]['patient_id']
p_ehr = ehr_df.iloc[0].to_dict()
p_wearable = wearable_df[wearable_df['patient_id'] == p_id].sort_values('timestamp')

print(f"Patient Profile for {p_id}:")
print(p_ehr)

plt.figure(figsize=(12, 4))
plt.plot(pd.to_datetime(p_wearable['timestamp'][:192]), p_wearable['glucose'][:192], label='Glucose (mg/dL)', color='#0284c7')
plt.axhline(140, color='r', linestyle='--', label='High Threshold (140 mg/dL)')
plt.title(f"2-Day Continuous Glucose Monitor (CGM) Stream — {p_id}")
plt.ylabel("Glucose (mg/dL)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 3. Digital Twin Instantiation & Prediction
twin = DigitalTwin(patient_id=p_id, ehr_profile=p_ehr, state_history=p_wearable.iloc[:48])
state = twin.get_current_state()
print("Digital Twin Current State:")
print(state)

pred = twin.predict()
print("Spike Risk Prediction:")
print(pred)

In [ ]:
# 4. What-If Simulation
sim_res = twin.simulate({"meal_carbs": 90.0, "steps": 0})
print("Simulation Result:")
print(sim_res)